# Preprocessing Pipeline with tf.data

This notebook builds the tf.data streaming pipeline:
- Dynamic resizing (all images → fixed tensor size)
- Rescaling pixel values to [0, 1]
- Isolated data augmentation (training only, prevents data leakage)
- Caching & prefetching (AUTOTUNE) to avoid I/O bottlenecks

In [ ]:
import tensorflow as tf
from pathlib import Path
from src.pipeline.data_pipeline import get_train_val_test, build_dataset

AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 32
IMG_SIZE = (150, 150)

# Build all three splits
train_ds, val_ds, test_ds = get_train_val_test(
    data_dir="../data/processed",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

# Count batches (materializes the pipeline — in real use, just iterate)
train_batches = sum(1 for _ in train_ds)
val_batches = sum(1 for _ in val_ds)
test_batches = sum(1 for _ in test_ds)

print(f"Train batches: {train_batches} ({train_batches * BATCH_SIZE} images)")
print(f"Val batches:   {val_batches} ({val_batches * BATCH_SIZE} images)")
print(f"Test batches:  {test_batches} ({test_batches * BATCH_SIZE} images)")

# Extract class names from directory structure
class_names = sorted([
    p.name for p in Path("../data/processed/train").iterdir()
    if p.is_dir()
])
print(f"Classes: {class_names}")

### Pipeline Inspection — Visualize a Batch

In [ ]:
import matplotlib.pyplot as plt

# Rebuild with small batch for display
ds_viz = build_dataset(
    "../data/processed/train",
    image_size=IMG_SIZE,
    batch_size=9,
    augment=True,
    shuffle=True,
)
images, labels = next(iter(ds_viz))

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].numpy())
    label_idx = labels[i].numpy()
    ax.set_title(class_names[label_idx] if label_idx < len(class_names) else str(label_idx))
    ax.axis("off")
plt.tight_layout()
plt.savefig("../reports/figures/augmentation_preview.png", dpi=150)
plt.show()
print("Saved augmentation preview to ../reports/figures/augmentation_preview.png")

### Pipeline Performance — Element Spec

In [ ]:
for images, labels in train_ds.take(1):
    print(f"Image batch shape: {images.shape}  dtype: {images.dtype}")
    print(f"Label batch shape: {labels.shape}  dtype: {labels.dtype}")
    print(f"Pixel range: [{tf.reduce_min(images):.3f}, {tf.reduce_max(images):.3f}]")
    break